# 🔍 RAG (Retrieval-Augmented Generation) dengan Groq
**Stack:** LangChain + ChromaDB + Groq (llama3-8b) + HuggingFace Embeddings

### Alur kerja:
```
PDF → Split Chunks → Embed → ChromaDB → Query → Retrieve → Groq LLM → Jawaban
```

## 📦 Cell 1 — Install Library

In [22]:
# Bersihkan semua paket langchain yang mungkin konflik
!pip uninstall -y langchain langchain-community langchain-core langchain-groq langchain-text-splitters langchain-huggingface

# Instalasi paksa menggunakan --no-cache-dir untuk menghindari file corrupt
!pip install --no-cache-dir -q -U langchain langchain-community langchain-core langchain-groq langchain-text-splitters langchain-huggingface chromadb pypdf

import site
from importlib import reload
reload(site)

print("✅ Instalasi selesai. Silakan RESTART SESSION sekarang!")

Found existing installation: langchain 1.3.11
Uninstalling langchain-1.3.11:
  Successfully uninstalled langchain-1.3.11
Found existing installation: langchain-community 0.4.2
Uninstalling langchain-community-0.4.2:
  Successfully uninstalled langchain-community-0.4.2
Found existing installation: langchain-core 1.4.8
Uninstalling langchain-core-1.4.8:
  Successfully uninstalled langchain-core-1.4.8
Found existing installation: langchain-groq 1.1.3
Uninstalling langchain-groq-1.1.3:
  Successfully uninstalled langchain-groq-1.1.3
Found existing installation: langchain-text-splitters 1.1.2
Uninstalling langchain-text-splitters-1.1.2:
  Successfully uninstalled langchain-text-splitters-1.1.2
Found existing installation: langchain-huggingface 1.2.2
Uninstalling langchain-huggingface-1.2.2:
  Successfully uninstalled langchain-huggingface-1.2.2
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 133.6/133.6 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 54.4 MB/s eta

## 🔑 Cell 2 — Set API Key Groq
> **Cara setup (sekali saja):**
> 1. Klik ikon 🔑 **kunci** di sidebar kiri Colab
> 2. Klik **"Add new secret"**
> 3. Name: `GROQ_API_KEY` → Value: API key lo
> 4. Toggle **"Notebook access"** → ON
> 5. Jalankan cell ini

In [23]:
import os
from google.colab import userdata

# Ambil API key dari Colab Secrets (aman, tidak tersimpan di notebook)
os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")

print("✅ API key berhasil di-load dari Colab Secrets!")

✅ API key berhasil di-load dari Colab Secrets!


## 📄 Cell 3 — Upload & Load PDF

In [24]:
from google.colab import files

print("📂 Pilih file PDF kamu:")
uploaded = files.upload()

# Ambil nama file yang diupload
pdf_filename = list(uploaded.keys())[0]
print(f"\n✅ File berhasil diupload: {pdf_filename}")

📂 Pilih file PDF kamu:


Saving final-uas-etika.pdf to final-uas-etika (11).pdf

✅ File berhasil diupload: final-uas-etika (11).pdf


## ✂️ Cell 4 — Load & Split Dokumen jadi Chunks

In [25]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Load PDF
loader = PyPDFLoader(pdf_filename)
pages = loader.load()
print(f"📖 Total halaman: {len(pages)}")

# Split jadi chunks
splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,      # max karakter per chunk
    chunk_overlap=50,    # overlap antar chunk biar konteks nyambung
)
chunks = splitter.split_documents(pages)

print(f"✂️  Total chunks: {len(chunks)}")
print(f"\n📝 Contoh chunk pertama:")
print(chunks[0].page_content)

📖 Total halaman: 5
✂️  Total chunks: 20

📝 Contoh chunk pertama:
Nama   : Muhammad Fadil 
Matkul   : Etika Profesi 
Program Studi  : Sistem Informasi 41 
Dosen Pengampu : Listiana Ningrum, M.T 
 
ESSAY HASIL WAWANCARA: MUCHAMAD FAHRUR RIZKY, S.KOM 
SEORANG TECHNICAL CONSULTANT PT. SOLUSI TIGA SELARAS (SOLUTIF) 
 
1. Profil dan Latar Belakang 
Muhammad Fahrur Rizky, narasumber dalam wawancara ini, adalah lulusan S1 
Sistem Informasi dari Institut Teknologi Kalimantan (ITK) dan saat ini bekerja


## 🧠 Cell 5 — Buat Embeddings & Simpan ke ChromaDB

In [26]:
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma

print("⏳ Loading embedding model (pertama kali agak lama ~1-2 menit)...")

# Model embedding gratis dari HuggingFace
embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

# Simpan chunks + embeddings ke ChromaDB (in-memory)
vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embedding_model,
)

print(f"✅ Vector store siap! Total vectors: {vectorstore._collection.count()}")

⏳ Loading embedding model (pertama kali agak lama ~1-2 menit)...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

✅ Vector store siap! Total vectors: 80


## 🤖 Cell 6 — Setup Groq LLM + RAG Chain

In [27]:
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough, RunnableLambda
from langchain_core.output_parsers import StrOutputParser

# 1. Inisialisasi LLM - Menggunakan model terbaru (llama-3.3-70b-versatile)
llm = ChatGroq(
    model_name="llama-3.3-70b-versatile",
    temperature=0.2,
)

# 2. Definisikan Prompt
system_prompt = (
    "Kamu adalah asisten untuk tugas menjawab pertanyaan. "
    "Gunakan potongan konteks berikut untuk menjawab pertanyaan. "
    "Jika kamu tidak tahu jawabannya, katakan bahwa kamu tidak tahu. "
    "Gunakan maksimal tiga kalimat dan jaga agar jawaban tetap ringkas.\n\n"
    "{context}"
)

prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    ("human", "{input}"),
])

# 3. Buat RAG Chain menggunakan LCEL
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

retriever = vectorstore.as_retriever()

rag_chain = (
    {
        "context": (lambda x: x["input"]) | retriever | format_docs,
        "input": lambda x: x["input"]
    }
    | prompt
    | llm
    | StrOutputParser()
)

# Fungsi pembungkus agar kompatibel dengan Cell 7 & 8
class LegacyChainWrapper:
    def __init__(self, chain, retriever):
        self.chain = chain
        self.retriever = retriever
    def invoke(self, inputs):
        docs = self.retriever.invoke(inputs["input"])
        answer = self.chain.invoke(inputs)
        return {"answer": answer, "context": docs}

rag_chain = LegacyChainWrapper(rag_chain, retriever)

print("✅ Model telah diupdate ke llama-3.3-70b-versatile! Silakan jalankan Cell 7.")

✅ Model telah diupdate ke llama-3.3-70b-versatile! Silakan jalankan Cell 7.


## 💬 Cell 7 — Tanya ke Dokumen!

In [28]:
# ✏️ Ganti pertanyaan di sini sesuai isi dokumen lo
pertanyaan = "Apa isi utama dari dokumen ini?"

print(f"❓ Pertanyaan: {pertanyaan}")
print("-" * 60)

# Menggunakan invoke dengan input key yang sesuai metode baru
result = rag_chain.invoke({"input": pertanyaan})

print(f"\n💡 Jawaban:\n{result['answer']}")

print("\n" + "=" * 60)
print("📚 Sumber chunks yang dipakai:")
for i, doc in enumerate(result['context'], 1):
    print(f"\n[Chunk {i}] Halaman {doc.metadata.get('page', '?') + 1}")
    print(doc.page_content[:200] + "...")

❓ Pertanyaan: Apa isi utama dari dokumen ini?
------------------------------------------------------------

💡 Jawaban:
Isi utama dari dokumen ini adalah tentang pentingnya membiasakan diri dengan bahasa Inggris sejak dini, terutama bagi mahasiswa yang ingin berkarier di bidang consultant. Narasumber juga berbagi pengalaman bahwa beliau sendiri belajar bahasa Inggris tidak melalui kelas formal, melainkan karena hobi main game berbahasa Inggris.

📚 Sumber chunks yang dipakai:

[Chunk 1] Halaman 3
Consultant 
Narasumber memberikan beberapa pesan yang cukup praktis dan personal bagi 
mahasiswa yang ingin berkarier di bidang ini. Yang pertama adalah soal bahasa Inggris. 
Beliau menekankan penting...

[Chunk 2] Halaman 3
Consultant 
Narasumber memberikan beberapa pesan yang cukup praktis dan personal bagi 
mahasiswa yang ingin berkarier di bidang ini. Yang pertama adalah soal bahasa Inggris. 
Beliau menekankan penting...

[Chunk 3] Halaman 3
Consultant 
Narasumber memberikan beberapa pesan y

## 🔁 Cell 8 — Mode Interaktif (Tanya Berulang)

In [29]:
print("💬 Mode tanya-jawab interaktif (ketik 'keluar' untuk stop)\n")

while True:
    pertanyaan = input("❓ Pertanyaan: ").strip()

    if pertanyaan.lower() in ["keluar", "exit", "quit"]:
        print("👋 Selesai!")
        break

    if not pertanyaan:
        continue

    # Menggunakan invoke dengan key 'input' sesuai rag_chain yang baru
    result = rag_chain.invoke({"input": pertanyaan})
    print(f"\n💡 Jawaban:\n{result['answer']}")
    print("-" * 60)

💬 Mode tanya-jawab interaktif (ketik 'keluar' untuk stop)

❓ Pertanyaan: siapa nama penulis dokumennya

💡 Jawaban:
Tidak disebutkan nama penulis dokumen tersebut.
------------------------------------------------------------
❓ Pertanyaan: nama dosennya

💡 Jawaban:
Tidak disebutkan nama dosen dalam teks tersebut.
------------------------------------------------------------
❓ Pertanyaan: siapa narasumber disitu

💡 Jawaban:
Tidak disebutkan secara eksplisit siapa narasumbernya, hanya disebut sebagai "Consultant" atau "Beliau" tanpa menyebutkan nama.
------------------------------------------------------------
❓ Pertanyaan: apa yg consultant kasih tau

💡 Jawaban:
Seorang Technical Consultant harus bersikap objektif dan jujur ketika memberikan rekomendasi kepada klien, serta menjelaskan kelebihan dan kekurangan dari setiap solusi teknologi yang tersedia.
------------------------------------------------------------
❓ Pertanyaan: keluar
👋 Selesai!


## 🔎 Cell 9 (Bonus) — Cek Similarity Search Manual

In [30]:
# Debug: lihat chunks apa yang diambil untuk query tertentu
query_debug = "Apa isi utama dari dokumen ini?"

docs = vectorstore.similarity_search(query_debug, k=3)

print(f"🔍 Top 3 chunks paling relevan untuk: '{query_debug}'\n")
for i, doc in enumerate(docs, 1):
    print(f"--- Chunk {i} (hal. {doc.metadata.get('page', '?') + 1}) ---")
    print(doc.page_content)
    print()

🔍 Top 3 chunks paling relevan untuk: 'Apa isi utama dari dokumen ini?'

--- Chunk 1 (hal. 3) ---
Consultant 
Narasumber memberikan beberapa pesan yang cukup praktis dan personal bagi 
mahasiswa yang ingin berkarier di bidang ini. Yang pertama adalah soal bahasa Inggris. 
Beliau menekankan pentingnya membiasakan diri dengan bahasa Inggris sejak dini, 
dan menariknya beliau sendiri tidak belajar lewat kelas formal. Awalnya karena hobi 
main game yang seluruhnya berbahasa Inggris, beliau  terpaksa mencari tahu artinya

--- Chunk 2 (hal. 3) ---
Consultant 
Narasumber memberikan beberapa pesan yang cukup praktis dan personal bagi 
mahasiswa yang ingin berkarier di bidang ini. Yang pertama adalah soal bahasa Inggris. 
Beliau menekankan pentingnya membiasakan diri dengan bahasa Inggris sejak dini, 
dan menariknya beliau sendiri tidak belajar lewat kelas formal. Awalnya karena hobi 
main game yang seluruhnya berbahasa Inggris, beliau  terpaksa mencari tahu artinya

--- Chunk 3 (hal. 3) ---
Con